<a href="https://colab.research.google.com/github/aousaf333/Projects-DS/blob/main/chapter_appendix-tools-for-deep-learning/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ==============================
# 2. LOAD DATA
# ==============================
train = pd.read_csv("/content/Housing-project-train-data.csv")
test = pd.read_csv("/content/Hosuing-project-test-data.csv")

train_id = train['Id']
test_id = test['Id']

train.drop("Id", axis=1, inplace=True)
test.drop("Id", axis=1, inplace=True)

# ==============================
# 3. HANDLE MISSING VALUES
# ==============================

# Fill numerical missing with median
num_cols = train.select_dtypes(include=['int64','float64']).columns
for col in num_cols:
    train[col].fillna(train[col].median(), inplace=True)
    if col in test.columns:
        test[col].fillna(test[col].median(), inplace=True)

# Fill categorical missing with 'None'
cat_cols = train.select_dtypes(include=['object']).columns
for col in cat_cols:
    train[col].fillna("None", inplace=True)
    if col in test.columns:
        test[col].fillna("None", inplace=True)

# ==============================
# 4. OUTLIER REMOVAL
# ==============================
train = train[train['GrLivArea'] < 4500]

# ==============================
# 5. TARGET TRANSFORMATION
# ==============================
train['SalePrice'] = np.log1p(train['SalePrice'])

# ==============================
# 6. FEATURE ENGINEERING
# ==============================

# Total square feet
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

# Total bathrooms
train['TotalBathrooms'] = train['FullBath'] + (0.5 * train['HalfBath'])
test['TotalBathrooms'] = test['FullBath'] + (0.5 * test['HalfBath'])

# ==============================
# 7. ENCODING
# ==============================
train = pd.get_dummies(train)
test = pd.get_dummies(test)

train, test = train.align(test, join='left', axis=1, fill_value=0)

# ==============================
# 8. SPLIT DATA
# ==============================
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ==============================
# 9. SCALING
# ==============================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
# Fix: Drop 'SalePrice' from the 'test' DataFrame before scaling it,
# as the scaler was fit on X_train which does not contain 'SalePrice'.
if 'SalePrice' in test.columns:
    test_features_for_scaling = test.drop('SalePrice', axis=1)
else:
    test_features_for_scaling = test
test_scaled = scaler.transform(test_features_for_scaling)

# ==============================
# 10. MODEL TRAINING
# ==============================

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(alpha=0.001),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(n_estimators=100)
}

results = {}

for name, model in models.items():
    if name in ["Linear", "Ridge", "Lasso"]:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_val_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)

    results[name] = (rmse, r2)
    print(f"{name} -> RMSE: {rmse:.4f}, R2: {r2:.4f}")

# ==============================
# 11. HYPERPARAMETER TUNING (RIDGE)
# ==============================
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 100]}
ridge = Ridge()

grid = GridSearchCV(ridge, ridge_params, cv=5)
grid.fit(X_train_scaled, y_train)

best_ridge = grid.best_estimator_

# Evaluate tuned model
ridge_preds = best_ridge.predict(X_val_scaled)

print("\nBest Ridge Model")
print("RMSE:", np.sqrt(mean_squared_error(y_val, ridge_preds)))
print("R2:", r2_score(y_val, ridge_preds))

# ==============================
# 12. FEATURE IMPORTANCE (LASSO)
# ==============================
lasso = Lasso(alpha=0.001)
lasso.fit(X_train_scaled, y_train)

importance = pd.Series(lasso.coef_, index=X.columns)
importance = importance.sort_values()

print("\nTop Positive Features:")
print(importance.tail(10))

print("\nTop Negative Features:")
print(importance.head(10))

# ==============================
# 13. FINAL MODEL (USE BEST RIDGE)
# ==============================
final_model = best_ridge
final_model.fit(X_train_scaled, y_train)

# ==============================
# 14. PREDICTIONS ON TEST DATA
# ==============================
test_preds = final_model.predict(test_scaled)

# Reverse log transform
test_preds = np.expm1(test_preds)

# ==============================
# 15. SUBMISSION FILE
# ==============================
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": test_preds
})

submission.to_csv("submission.csv", index=False)

print("\n✅ Submission file created successfully!")

/tmp/ipykernel_12001/556705670.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(train[col].median(), inplace=True)
/tmp/ipykernel_12001/556705670.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)'

Linear -> RMSE: 0.1265, R2: 0.9028
Ridge -> RMSE: 0.1262, R2: 0.9034
Lasso -> RMSE: 0.1114, R2: 0.9247
DecisionTree -> RMSE: 0.2164, R2: 0.7156
RandomForest -> RMSE: 0.1361, R2: 0.8875

Best Ridge Model
RMSE: 0.11339253065789613
R2: 0.9219328419086624

Top Positive Features:
KitchenQual_Ex          0.016653
Neighborhood_StoneBr    0.016788
YearRemodAdd            0.023636
BsmtFinSF1              0.024010
LotArea                 0.030385
OverallCond             0.034156
YearBuilt               0.045089
OverallQual             0.066582
TotalSF                 0.081594
GrLivArea               0.081917
dtype: float64

Top Negative Features:
MSZoning_C (all)        -0.038612
LandSlope_Sev           -0.022639
MSZoning_RM             -0.019424
Functional_Maj2         -0.018976
CentralAir_N            -0.016680
SaleCondition_Abnorml   -0.014967
Exterior1st_BrkComm     -0.014642
SaleType_WD             -0.013588
GarageType_None         -0.012133
Neighborhood_Edwards    -0.011910
dtype: float64


In [12]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ==========================================
# 2. LOAD YOUR FILES
# ==========================================
train = pd.read_csv("/content/Housing-project-train-data.csv")
test = pd.read_csv("/content/Hosuing-project-test-data.csv")

# Save Id for submission
train_id = train['Id']
test_id = test['Id']

train.drop("Id", axis=1, inplace=True)
test.drop("Id", axis=1, inplace=True)

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

# ==========================================
# 3. HANDLE MISSING VALUES (SMART WAY)
# ==========================================

# According to data description:
# Many NA = absence (Garage, Pool, Basement)

none_cols = [
    'Alley','MasVnrType','BsmtQual','BsmtCond','BsmtExposure',
    'BsmtFinType1','BsmtFinType2','FireplaceQu','GarageType',
    'GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature'
]

for col in none_cols:
    if col in train.columns:
        train[col].fillna("None", inplace=True)
    if col in test.columns:
        test[col].fillna("None", inplace=True)

# Numerical missing → median
num_cols = train.select_dtypes(include=['int64','float64']).columns
for col in num_cols:
    if train[col].isnull().sum() > 0:
        train[col].fillna(train[col].median(), inplace=True)
    if col in test.columns and test[col].isnull().sum() > 0:
        test[col].fillna(test[col].median(), inplace=True)

# Remaining categorical → mode
cat_cols = train.select_dtypes(include=['object']).columns
for col in cat_cols:
    train[col].fillna(train[col].mode()[0], inplace=True)
    if col in test.columns:
        test[col].fillna(test[col].mode()[0], inplace=True)

# ==========================================
# 4. OUTLIER REMOVAL
# ==========================================
train = train[train['GrLivArea'] < 4500]

# ==========================================
# 5. TARGET TRANSFORMATION
# ==========================================
train['SalePrice'] = np.log1p(train['SalePrice'])

# ==========================================
# 6. FEATURE ENGINEERING
# ==========================================

# Total Square Feet
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

# Total Bathrooms
train['TotalBathrooms'] = (
    train['FullBath'] + 0.5*train['HalfBath'] +
    train['BsmtFullBath'] + 0.5*train['BsmtHalfBath']
)

test['TotalBathrooms'] = (
    test['FullBath'] + 0.5*test['HalfBath'] +
    test['BsmtFullBath'] + 0.5*test['BsmtHalfBath']
)

# House Age
train['HouseAge'] = train['YrSold'] - train['YearBuilt']
test['HouseAge'] = test['YrSold'] - test['YearBuilt']

# ==========================================
# 7. ENCODING
# ==========================================
train = pd.get_dummies(train)
test = pd.get_dummies(test)

# Align columns
train, test = train.align(test, join='left', axis=1, fill_value=0)

# ==========================================
# 8. SPLIT DATA
# ==========================================
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==========================================
# 9. SCALING
# ==========================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Fix: Drop 'SalePrice' from the 'test' DataFrame before scaling it,
# as the scaler was fit on X_train which does not contain 'SalePrice'.
if 'SalePrice' in test.columns:
    test_features_for_scaling = test.drop('SalePrice', axis=1)
else:
    test_features_for_scaling = test
test_scaled = scaler.transform(test_features_for_scaling)

# ==========================================
# 10. MODEL TRAINING
# ==========================================
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(alpha=0.001),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42)
}

results = {}

print("\nMODEL PERFORMANCE:\n")

for name, model in models.items():
    if name in ["Linear", "Ridge", "Lasso"]:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_val_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)

    results[name] = (rmse, r2)
    print(f"{name} -> RMSE: {rmse:.4f}, R2: {r2:.4f}")

# ==========================================
# 11. HYPERPARAMETER TUNING (RIDGE)
# ==========================================
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 100]}

grid = GridSearchCV(Ridge(), ridge_params, cv=5)
grid.fit(X_train_scaled, y_train)

best_ridge = grid.best_estimator_

print("\nBest Ridge Alpha:", grid.best_params_)

ridge_preds = best_ridge.predict(X_val_scaled)

print("Tuned Ridge RMSE:",
      np.sqrt(mean_squared_error(y_val, ridge_preds)))
print("Tuned Ridge R2:",
      r2_score(y_val, ridge_preds))

# ==========================================
# 12. FEATURE IMPORTANCE (LASSO)
# ==========================================
lasso = Lasso(alpha=0.001)
lasso.fit(X_train_scaled, y_train)

importance = pd.Series(lasso.coef_, index=X.columns)
importance = importance.sort_values()

print("\nTop Positive Features:\n", importance.tail(10))
print("\nTop Negative Features:\n", importance.head(10))

# ==========================================
# 13. FINAL MODEL
# ==========================================
final_model = best_ridge
final_model.fit(X_train_scaled, y_train)

# ==========================================
# 14. TEST PREDICTIONS
# ==========================================
test_preds = final_model.predict(test_scaled)

# Reverse log transform
test_preds = np.expm1(test_preds)

# ==========================================
# 15. SAVE SUBMISSION
# ==========================================
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": test_preds
})

submission.to_csv("final_submission.csv", index=False)

print("\n✅ final_submission.csv created successfully!")

Train Shape: (1168, 80)
Test Shape: (292, 79)


/tmp/ipykernel_12001/495373391.py:47: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna("None", inplace=True)
/tmp/ipykernel_12001/495373391.py:49: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using '


MODEL PERFORMANCE:

Linear -> RMSE: 0.1265, R2: 0.9028
Ridge -> RMSE: 0.1262, R2: 0.9033
Lasso -> RMSE: 0.1115, R2: 0.9245
DecisionTree -> RMSE: 0.2116, R2: 0.7281
RandomForest -> RMSE: 0.1365, R2: 0.8868

Best Ridge Alpha: {'alpha': 100}
Tuned Ridge RMSE: 0.11370373417693748
Tuned Ridge R2: 0.9215037464040803

Top Positive Features:
 Neighborhood_Crawfor    0.015995
KitchenQual_Ex          0.016730
Neighborhood_StoneBr    0.016873
YearRemodAdd            0.023534
BsmtFinSF1              0.023946
LotArea                 0.030274
OverallCond             0.034246
OverallQual             0.066569
GrLivArea               0.078674
TotalSF                 0.082433
dtype: float64

Top Negative Features:
 MSZoning_C (all)        -0.038433
HouseAge                -0.034969
LandSlope_Sev           -0.022409
MSZoning_RM             -0.018935
Functional_Maj2         -0.018747
CentralAir_N            -0.016696
SaleCondition_Abnorml   -0.015084
Exterior1st_BrkComm     -0.014516
SaleType_WD         